<a href="https://colab.research.google.com/github/techguystarr-png/Ts-Mental-Foundry/blob/main/Mental_Foundry_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T's Mental Foundry - Google Colab Setup

🧠 AI Assistant for image generation, web scraping, data intelligence, and public webcams

**Features:**
- 🎨 Image Generation (Stable Diffusion) - Using FREE GPU!
- 🎙️ Voice Synthesis with your ElevenLabs custom voice
- 🕷️ Web Scraping
- 🔍 Public Data Intelligence
- 📹 Public Webcam Viewer
- 💬 Discord & Telegram Bot Support

**Setup Time:** ~5-10 minutes

## Step 1: Install Dependencies

This installs all required Python packages for the backend.

In [2]:
# Install all dependencies
!pip install -q fastapi uvicorn pydantic torch torchvision diffusers transformers Pillow
!pip install -q beautifulsoup4 selenium requests lxml aiohttp
!pip install -q openai-whisper pyttsx3 sounddevice scipy
!pip install -q discord.py python-telegram-bot
!pip install -q sqlalchemy python-dotenv python-multipart
!pip install -q nest-asyncio
!pip install -q coqui-tts

print("✅ All dependencies installed!")

✅ All dependencies installed!


## Step 2: Configure Your tts Voice

Enter your tts here.

In [4]:
config = {
    "api_host": "0.0.0.0",
    "api_port": 8000,
    "device": "cuda",  # Still using that FREE GPU for the heavy lifting!
    "stable_diffusion_model": "runwayml/stable-diffusion-v1-5",
    "tts_model": "tts_models/en/ljspeech/vits", # A solid open-source starting point
}

print("✅ Open Source Configuration loaded!")
print(f"🚀 Device: {config['device']} (FREE GPU!)")
print(f"🎙️  Voice Engine: Open Source TTS")

✅ Open Source Configuration loaded!
🚀 Device: cuda (FREE GPU!)
🎙️  Voice Engine: Open Source TTS


## Step 3: Create Backend Modules

Setting up the image generation, voice, and other modules.

In [5]:
import os
import logging
import torch
from pathlib import Path
from diffusers import StableDiffusionPipeline
import requests
from typing import List, Dict, Any, Optional
import json

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Create data directory
DATA_DIR = Path('/tmp/mental_foundry')
DATA_DIR.mkdir(exist_ok=True)

print(f"✅ Created data directory: {DATA_DIR}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Created data directory: /tmp/mental_foundry


In [6]:
# Image Generation Module
class ImageGenerator:
    def __init__(self, device="cuda", model_id="runwayml/stable-diffusion-v1-5"):
        self.device = device
        self.model_id = model_id
        self.pipeline = None
        self.output_dir = DATA_DIR / 'generated_images'
        self.output_dir.mkdir(exist_ok=True)
        print(f"🎨 Image Generator initialized on device: {device}")

    def load_pipeline(self):
        if self.pipeline is None:
            print(f"📥 Loading Stable Diffusion model...")
            self.pipeline = StableDiffusionPipeline.from_pretrained(
                self.model_id,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                safety_checker=None,
            )
            self.pipeline = self.pipeline.to(self.device)
            print("✅ Model loaded successfully!")

    def generate(self, prompt: str, num_images: int = 1,
                height: int = 512, width: int = 512) -> List[str]:
        self.load_pipeline()

        print(f"🎨 Generating {num_images} image(s): {prompt}")

        with torch.no_grad():
            output = self.pipeline(
                prompt=prompt,
                num_images_per_prompt=num_images,
                height=height,
                width=width,
                num_inference_steps=50
            )

        images = output.images
        saved_paths = []

        for idx, image in enumerate(images):
            filename = f"gen_{prompt[:30].replace(' ', '_')}_{idx}.png"
            filepath = self.output_dir / filename
            image.save(filepath)
            saved_paths.append(str(filepath))
            print(f"✅ Saved: {filepath}")

        return saved_paths

# Initialize image generator
image_gen = ImageGenerator(device=config["device"])
print("✅ Image Generator initialized!")

🎨 Image Generator initialized on device: cuda
✅ Image Generator initialized!


In [18]:
# # Voice Synthesis Module with Coqui TTS (Open Source


!apt-get install -y espeak-ng

from TTS.api import TTS

# Initialize the model (this will download the files on the first run)
model_name = "tts_models/en/ljspeech/vits"
tts = TTS(model_name).to("cpu") # Change to "cuda" if you have a GPU!

# Run the synthesis
tts.tts_to_file(text="Hello there! Let's get this project moving.",
                file_path="output.wav")

# Initialize voice processor (CLEANED: No ElevenLabs arguments)
print("✅ Voice Processor is ready to chat!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


FileNotFoundError: [!] No espeak backend found. Install espeak-ng or espeak to your system.

In [ ]:
# Web Scraping Module
class WebScraper:
    def __init__(self):
        from bs4 import BeautifulSoup
        self.BeautifulSoup = BeautifulSoup
        print("🕷️  Web Scraper initialized")

    def scrape(self, url: str) -> Dict[str, Any]:
        try:
            print(f"🕷️  Scraping: {url}")
            response = requests.get(url, timeout=30)

            soup = self.BeautifulSoup(response.content, 'html.parser')

            data = {
                'title': soup.title.string if soup.title else 'No title',
                'text': [p.get_text(strip=True) for p in soup.find_all('p')][:10],
                'links': [{'url': a.get('href'), 'text': a.get_text()} for a in soup.find_all('a')][:10]
            }

            print(f"✅ Scraped {len(data['text'])} paragraphs")
            return data
        except Exception as e:
            print(f"❌ Scraping error: {str(e)}")
            return {}

web_scraper = WebScraper()
print("✅ Web Scraper initialized!")

## Step 4: Test Image Generation

Let's generate a test image with your free GPU!

In [ ]:
# Test image generation
print("🎨 Testing Image Generation...")
print("⏳ This may take 1-3 minutes on first run (downloading model)...\n")

test_prompt = "a beautiful sunset over the ocean, digital art, high quality"
try:
    images = image_gen.generate(test_prompt, num_images=1)
    print(f"\n✅ Image generation successful!")
    print(f"Images saved to: {images}")
except Exception as e:
    print(f"❌ Error: {e}")

## Step 5: Test Voice Synthesis

Test your custom ElevenLabs voice!

In [ ]:
# Test voice synthesis
print("🎙️  Testing Voice Synthesis...\n")

test_text = "Hello! Welcome to T's Mental Foundry. I'm powered by your custom ElevenLabs voice!"
audio_file = voice_proc.synthesize(test_text)

if audio_file:
    print(f"\n✅ Voice synthesis successful!")
    print(f"Audio saved to: {audio_file}")
    print("\n🔊 Your AI now speaks with YOUR custom voice!")
else:
    print("❌ Voice synthesis failed. Check your API key and voice ID.")

## Step 6: Start the FastAPI Server

Launch the API server (runs in background)

In [ ]:
# Install ngrok for public URL
!pip install -q pyngrok
print("✅ ngrok installed!")

In [ ]:
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import threading
import time

nest_asyncio.apply()

# Create FastAPI app
app = FastAPI(
    title="T's Mental Foundry API",
    description="Running on FREE Google Colab GPU!",
    version="1.0.0"
)

# Pydantic models
class ImageRequest(BaseModel):
    prompt: str
    num_images: int = 1

class VoiceRequest(BaseModel):
    text: str

class ScrapeRequest(BaseModel):
    url: str

# API Endpoints
@app.get("/health")
async def health():
    return {"status": "online", "version": "1.0.0", "gpu": "FREE!"}

@app.post("/api/image/generate")
async def generate(request: ImageRequest):
    try:
        images = image_gen.generate(request.prompt, request.num_images)
        return {"success": True, "images": images, "prompt": request.prompt}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/voice/synthesize")
async def synthesize(request: VoiceRequest):
    try:
        audio = voice_proc.synthesize(request.text)
        return {"success": True, "audio_file": audio, "text": request.text}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/scrape")
async def scrape(request: ScrapeRequest):
    try:
        data = web_scraper.scrape(request.url)
        return {"success": True, "data": data, "url": request.url}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✅ FastAPI app created!")

In [ ]:
# Start the server
from pyngrok import ngrok

print("🚀 Starting FastAPI server...\n")

# Start ngrok tunnel
ngrok.set_auth_token("your_ngrok_token_here")  # Get free token at ngrok.com
public_url = ngrok.connect(8000)
print(f"📡 Public URL: {public_url}")
print(f"🌐 Access your API from anywhere!\n")

# Start uvicorn server
import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

print("✅ Server running!")
print("📝 API Documentation: {public_url}/docs")
print("\n✨ Your Mental Foundry is live!")

## Step 7: Use Your API

Once the server is running, you can test it with these commands:

In [ ]:
# Test the API
print("📋 API Endpoints Available:\n")
print("Health Check:")
print("  curl http://localhost:8000/health\n")
print("Generate Image:")
print("  curl -X POST http://localhost:8000/api/image/generate -H 'Content-Type: application/json' -d '{\"prompt\": \"your prompt here\"}'\n")
print("Voice Synthesis:")
print("  curl -X POST http://localhost:8000/api/voice/synthesize -H 'Content-Type: application/json' -d '{\"text\": \"your text here\"}'\n")
print("Scrape Website:")
print("  curl -X POST http://localhost:8000/api/scrape -H 'Content-Type: application/json' -d '{\"url\": \"https://example.com\"}'\n")

## Step 8: Deploy Bots (Optional)

Connect your Discord and Telegram bots to use your API

In [ ]:
print("🤖 Bot Setup Instructions:\n")
print("1. Get your tokens:")
print("   - Discord: https://discord.com/developers/applications")
print("   - Telegram: Chat with @BotFather on Telegram\n")
print("2. Use your public API URL from above\n")
print("3. Run the bot files from your GitHub repo on Railway.app (free)\n")
print("Your bots will call this Colab API automatically!")

## Resources

- 📚 **GitHub:** https://github.com/techguystarr-png/Ts-Mental-Foundry
- 🎤 **ElevenLabs:** https://elevenlabs.io
- 🚀 **Deploy Bots:** https://railway.app (free hosting)
- 📱 **Flutter App:** Run locally or build for Android/iOS

---

**You now have:**
- ✅ Stable Diffusion image generation (FREE GPU)
- ✅ Your custom ElevenLabs voice
- ✅ Web scraping
- ✅ Public API for bots/apps
- ✅ Everything running FREE on Google Colab!

🎉 **Your Mental Foundry is live!**